# Cowork 태스크 루프 시뮬레이션 — Task Loop Simulation

**Introduction to Claude Cowork — L01~L03 대응**

이 노트북에서 다루는 내용:
1. Cowork의 핵심 태스크 루프(Describe → Plan → Execute → Review)를 API로 시뮬레이션
2. 서브에이전트 병렬 처리 개념 체험
3. Chat vs Cowork 방식의 차이 비교

> **참고**: 실제 Cowork는 Claude Desktop의 GUI 기능입니다. 이 노트북은 동일한 패턴을 API로 재현하여 원리를 이해합니다.

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import asyncio
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. Chat 방식 vs Cowork 방식 비교

**Chat 방식**: 사용자가 텍스트를 붙여넣고, 답변을 받고, 다시 복사한다.
**Cowork 방식**: 사용자가 작업을 설명하면, Claude가 계획을 세우고, 실행하고, 파일을 생성한다.

API로 두 방식의 차이를 체험해봅시다.

In [ ]:
# Chat 방식: 단일 질문-답변
def chat_style(question: str) -> str:
    """Chat 방식 — 질문을 보내고 텍스트 답변을 받는다."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text

# 예시: 회의록 요약 요청
result = chat_style(
    "다음 회의록을 요약해줘:\n"
    "- 참석자: 김팀장, 이대리, 박과장\n"
    "- 안건: 2분기 예산 검토\n"
    "- 결정사항: 마케팅 예산 10% 증액, IT 장비 구매 보류\n"
    "- 다음 회의: 4/20"
)
print("=== Chat 방식 결과 ===")
print(result)
print("\n→ 결과를 복사해서 문서에 붙여넣어야 합니다.")

## §2. Cowork 태스크 루프 시뮬레이션

Cowork의 4단계 루프를 API로 재현합니다:
1. **Describe** — 사용자가 작업을 설명
2. **Plan** — Claude가 계획을 수립하고 사용자에게 보여줌
3. **Execute** — 계획에 따라 단계별 실행
4. **Review** — 결과를 파일로 저장

In [ ]:
def cowork_plan(task_description: str, file_list: list[str]) -> dict:
    """Step 1-2: Describe + Plan — 작업 설명을 받고 실행 계획을 수립한다."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=(
            "당신은 Claude Cowork의 계획 단계를 시뮬레이션합니다.\n"
            "사용자의 작업 설명과 가용 파일 목록을 받아 실행 계획을 JSON으로 작성합니다.\n"
            "JSON 형식: {\"plan_title\": \"...\", \"steps\": [{\"step\": 1, \"action\": \"...\", \"files\": [...]}], "
            "\"output_file\": \"...\", \"estimated_time\": \"...\"}\n"
            "반드시 유효한 JSON만 출력하세요."
        ),
        messages=[{
            "role": "user",
            "content": (
                f"작업 설명: {task_description}\n\n"
                f"폴더 내 파일 목록:\n" +
                "\n".join(f"  - {f}" for f in file_list)
            )
        }]
    )
    return json.loads(response.content[0].text)

# 시뮬레이션: 프로젝트 폴더의 파일 목록
files = [
    "meeting_notes_0401.txt",
    "meeting_notes_0408.txt",
    "budget_q2.xlsx",
    "vendor_comparison.csv",
    "project_timeline.xlsx"
]

task = "이 폴더의 회의록과 예산 데이터를 분석하여 Q2 프로젝트 현황 보고서를 작성해줘."

# Step 1-2: 계획 수립
plan = cowork_plan(task, files)
print("=== Cowork 실행 계획 ===")
print(json.dumps(plan, indent=2, ensure_ascii=False))

In [ ]:
def cowork_execute(plan: dict) -> str:
    """Step 3: Execute — 계획의 각 단계를 순차적으로 실행한다."""
    steps = plan.get("steps", [])
    results = []
    
    for step in steps:
        step_num = step.get("step", "?")
        action = step.get("action", "")
        files_used = step.get("files", [])
        
        print(f"  ⚙️ Step {step_num}: {action}")
        if files_used:
            print(f"     📂 파일: {', '.join(files_used)}")
        
        # 각 단계를 Claude에게 실행 요청
        response = client.messages.create(
            model=MODEL,
            max_tokens=512,
            system="당신은 Cowork의 실행 단계를 시뮬레이션합니다. 각 단계의 결과를 간결하게 작성하세요.",
            messages=[{
                "role": "user",
                "content": f"다음 단계를 실행하세요: {action}\n사용 파일: {files_used}"
            }]
        )
        result = response.content[0].text
        results.append(f"### Step {step_num}: {action}\n{result}")
        print(f"     ✅ 완료")
    
    return "\n\n".join(results)

# Step 3: 실행
print("=== Cowork 실행 중 ===")
execution_result = cowork_execute(plan)
print("\n=== 실행 완료 ===")

In [ ]:
# Step 4: Review — 결과를 파일로 저장하고 확인
import os

output_dir = "cowork_output"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, plan.get("output_file", "report.md"))
with open(output_file, "w", encoding="utf-8") as f:
    f.write(f"# {plan.get('plan_title', 'Report')}\n\n")
    f.write(f"생성일: 2026-04-13\n\n")
    f.write(execution_result)

print(f"=== Cowork 결과 파일 저장 완료 ===")
print(f"📄 파일: {output_file}")
print(f"📊 크기: {os.path.getsize(output_file):,} bytes")
print(f"\n→ 실제 Cowork에서는 PPT, XLSX, PDF 등 네이티브 파일로 생성됩니다.")
print(f"→ Chat과 달리 복사-붙여넣기 없이 바로 사용 가능합니다.")

## §3. 서브에이전트 병렬 처리 시뮬레이션

Cowork의 핵심 기능 중 하나: 독립적인 작업을 **서브에이전트**로 동시에 처리합니다.
각 서브에이전트는 자체 컨텍스트를 가지므로, 다른 작업의 세부사항에 의해 분석이 희석되지 않습니다.

In [ ]:
async def subagent_analyze(agent_name: str, vendor_info: str) -> dict:
    """서브에이전트 — 각 벤더를 독립적으로 분석한다."""
    async_client = anthropic.AsyncAnthropic()
    
    response = await async_client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=(
            f"당신은 '{agent_name}' 서브에이전트입니다.\n"
            "주어진 벤더 정보를 분석하여 JSON으로 결과를 반환하세요.\n"
            '형식: {"vendor": "...", "score": N, "pros": [...], "cons": [...], "recommendation": "..."}\n'
            "반드시 유효한 JSON만 출력하세요."
        ),
        messages=[{"role": "user", "content": f"다음 벤더를 분석하세요:\n{vendor_info}"}]
    )
    result = json.loads(response.content[0].text)
    print(f"  ✅ {agent_name} 분석 완료: {result.get('vendor', '?')} (점수: {result.get('score', '?')})")
    return result


async def cowork_parallel_analysis():
    """Cowork 스타일 병렬 분석 — 서브에이전트가 동시에 작업한다."""
    vendors = [
        "벤더 A: 클라우드 서비스, 월 500만원, SLA 99.9%, 국내 데이터센터, 24/7 지원",
        "벤더 B: 하이브리드 솔루션, 월 350만원, SLA 99.5%, 해외 데이터센터, 평일 지원",
        "벤더 C: 온프레미스, 초기 3억원, SLA 99.99%, 자체 운영, 커스터마이즈 가능",
    ]
    
    print("=== 서브에이전트 병렬 분석 시작 ===")
    print(f"  📋 {len(vendors)}개 벤더를 동시에 분석합니다.\n")
    
    # 서브에이전트를 동시에 실행 (asyncio.gather = Week 09 본편 병렬화)
    tasks = [
        subagent_analyze(f"Agent-{i+1}", vendor)
        for i, vendor in enumerate(vendors)
    ]
    results = await asyncio.gather(*tasks)
    
    print(f"\n=== 분석 완료 — {len(results)}개 결과 통합 ===")
    return results

# 실행 (Jupyter에서는 await 직접 사용 가능)
results = await cowork_parallel_analysis()
print("\n=== 통합 결과 ===")
print(json.dumps(results, indent=2, ensure_ascii=False))

## §4. 핵심 정리

- **Chat**: 질문 → 답변 → 복사. 단일 대화 내에서 작동.
- **Cowork**: Describe → Plan → Execute → Review. 파일 시스템에서 직접 작업하고 결과를 파일로 저장.
- **서브에이전트**: 독립적인 작업을 병렬로 처리. 각 에이전트가 자체 컨텍스트를 가짐 (Week 09 본편의 `asyncio.gather` 병렬화와 동일 원리).
- **다음 노트북**: `CW_02_skills_and_plugins.ipynb`에서 스킬과 플러그인 시스템을 실습합니다.